# Demonstração — Chamar uma API de LLM

Este notebook mostra como interagir com um Large Language Model (LLM) através de uma API, utilizando fornecedores **gratuitos**.

---

## O que vamos aprender?

1. Instalar e configurar a biblioteca necessária
2. Compreender a estrutura de um pedido à API (messages, roles)
3. Controlar o comportamento do modelo (temperature, max_tokens)
4. Exemplos práticos: prompt simples, system prompt, multi-turno
5. Comparar o efeito de diferentes parâmetros

---

## Pré-requisitos

Vamos usar o **Groq** como fornecedor principal — é gratuito, muito rápido (>300 tokens/segundo) e tem limites generosos.

1. Aceder a [console.groq.com](https://console.groq.com/)
2. Criar uma conta (pode ser com Google)
3. Ir a **API Keys** → **Create API Key**
4. Copiar a chave e colá-la na célula de configuração abaixo

> **Nota:** não precisa de cartão de crédito. Os limites gratuitos são suficientes para esta demonstração.

---
## 1. Instalação

Utilizamos a biblioteca `openai` do Python. Apesar do nome, esta biblioteca pode comunicar com **qualquer API compatível com o formato OpenAI** — incluindo Groq, Google Gemini, e outros.

### Porquê a biblioteca `openai`?
O formato de API da OpenAI tornou-se um **padrão da indústria**. A maioria dos fornecedores oferece endpoints compatíveis, o que permite trocar de modelo mudando apenas a `base_url` e o `model`.

In [ ]:
# Instalar a biblioteca openai (executar apenas uma vez)
!pip install openai -q

---
## 2. Configuração do Cliente

Vamos criar um **cliente** — o objeto que faz a ponte entre o nosso código e a API do LLM.

### Parâmetros do `OpenAI()`

| Parâmetro | Tipo | Descrição |
|---|---|---|
| `api_key` | `str` | A chave secreta que identifica a tua conta. Nunca partilhes esta chave publicamente. |
| `base_url` | `str` | O endereço do servidor da API. Ao mudar este valor, podemos apontar para diferentes fornecedores (Groq, Google, OpenAI, etc.) sem alterar o resto do código. |

In [ ]:
from openai import OpenAI

# ============================================================
# CONFIGURAÇÃO — Coloca aqui a tua API Key
# ============================================================

#   Obtém a chave em: https://console.groq.com/keys
API_KEY  = "COLOQUE SUA CHAVE AQUI"
BASE_URL = "https://api.groq.com/openai/v1"
MODELO   = "llama-3.3-70b-versatile"  # Modelo Llama 3.3 70B (gratuito no Groq)

# ============================================================
# Criar o cliente
# ============================================================
client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print(f"✅ Cliente configurado!")
print(f"   Fornecedor: {BASE_URL}")
print(f"   Modelo:     {MODELO}")

---
## 3. O Primeiro Pedido — Conceitos Fundamentais

Cada pedido à API usa o método `client.chat.completions.create()`. Vamos entender cada parâmetro.

### Parâmetros de `chat.completions.create()`

| Parâmetro | Tipo | Default | Descrição |
|---|---|---|---|
| `model` | `str` | — (obrigatório) | Nome do modelo a utilizar. Cada fornecedor tem os seus modelos. |
| `messages` | `list[dict]` | — (obrigatório) | Lista de mensagens que compõem a conversa. Cada mensagem é um dicionário com `role` e `content`. **Este é o parâmetro mais importante.** |
| `temperature` | `float` | `1.0` | Controla a aleatoriedade. De `0` (determinístico) a `2` (muito criativo). |
| `max_tokens` | `int` | varia | Número máximo de tokens que o modelo pode gerar na resposta. |
| `top_p` | `float` | `1.0` | Alternativa ao temperature — nucleus sampling. Considera apenas os tokens mais prováveis cuja probabilidade acumulada atinge `p`. Valor entre `0` e `1`. |
| `n` | `int` | `1` | Número de respostas alternativas a gerar. |
| `stop` | `str` ou `list` | `None` | Sequência(s) onde o modelo deve parar de gerar. Ex: `["\n"]` para parar no fim da linha. |
| `stream` | `bool` | `False` | Se `True`, a resposta é enviada token a token (em tempo real). |

### O que são `messages`?

As `messages` são uma **lista de mensagens** que representam a conversa. Cada mensagem é um dicionário com dois campos:

- **`role`** — quem está a falar:

| Role | Quem é | Para que serve | Exemplo |
|---|---|---|---|
| `"system"` | O programador | Instruções globais invisíveis ao utilizador final. Define quem o modelo é e como se comporta. | `"És um nutricionista. Responde em português."` |
| `"user"` | O utilizador | A pergunta ou pedido do utilizador. | `"Que alimentos têm ferro?"` |
| `"assistant"` | O modelo | Resposta anterior do modelo. Usada para manter o contexto em conversas multi-turno. | `"Espinafres, lentilhas e carne vermelha."` |

- **`content`** — o texto da mensagem (`str`)

⚠️ **O modelo não tem memória entre pedidos.** Para continuar uma conversa, é necessário enviar todo o histórico em cada pedido.

In [ ]:
# ============================================================
# EXEMPLO 1 — Pedido simples (apenas user)
# ============================================================

resposta = client.chat.completions.create(
    model=MODELO,                 # Modelo definido na configuração
    messages=[
        {
            "role": "user",        # Quem fala: o utilizador
            "content": "O que é inteligência artificial? Responde em 2 frases."
        }
    ]
)

# Extrair o texto da resposta
texto = resposta.choices[0].message.content
print(texto)

### Compreender o objeto de resposta

A API devolve um objeto estruturado. Os campos mais relevantes são:

| Campo | Tipo | Descrição |
|---|---|---|
| `resposta.choices` | `list` | Lista de respostas geradas (normalmente 1). |
| `resposta.choices[0].message.content` | `str` | O texto da resposta do modelo. |
| `resposta.choices[0].message.role` | `str` | O role da resposta (sempre `"assistant"`). |
| `resposta.choices[0].finish_reason` | `str` | Razão pela qual o modelo parou: `"stop"` (terminou naturalmente) ou `"length"` (atingiu o `max_tokens`). |
| `resposta.usage.prompt_tokens` | `int` | Número de tokens enviados (entrada). É sobre estes que se paga o custo de input. |
| `resposta.usage.completion_tokens` | `int` | Número de tokens gerados (saída). É sobre estes que se paga o custo de output. |
| `resposta.usage.total_tokens` | `int` | Total de tokens = entrada + saída. |

In [ ]:
# ============================================================
# Explorar o objeto de resposta
# ============================================================

print(f"Modelo utilizado:   {resposta.model}")
print(f"Razão de paragem:   {resposta.choices[0].finish_reason}")
print()
print("--- Utilização de tokens ---")
print(f"Tokens de entrada:  {resposta.usage.prompt_tokens}")
print(f"Tokens de saída:    {resposta.usage.completion_tokens}")
print(f"Total de tokens:    {resposta.usage.total_tokens}")
print()
print("💡 Numa API paga, o custo seria calculado com base nestes valores.")
print(f"   Exemplo com Claude Sonnet 4.5: "
      f"${resposta.usage.prompt_tokens * 3 / 1_000_000:.6f} (input) + "
      f"${resposta.usage.completion_tokens * 15 / 1_000_000:.6f} (output)")

---
## 4. System Prompt — Definir o Papel do Modelo

O `system` prompt é uma mensagem especial que define **quem o modelo é** e **como se deve comportar**. É invisível para o utilizador final, mas influencia todas as respostas.

É aqui que aplicamos conceitos de **engenharia de prompts**:
- Definir um papel (ex: "és um nutricionista")
- Definir o formato de resposta (ex: "responde em tópicos")
- Definir restrições (ex: "máximo 3 frases", "em português de Portugal")

In [ ]:
# ============================================================
# EXEMPLO 2 — Usar system prompt para definir o papel
# ============================================================

resposta = client.chat.completions.create(
    model=MODELO,
    messages=[
        {
            "role": "system",       # ← Instruções invisíveis para o utilizador final
            "content": (
                "És um nutricionista simpático e experiente. "
                "Respondes sempre em português de Portugal. "
                "Usas linguagem simples e acessível. "
                "Incluís sempre uma dica prática no final."
            )
        },
        {
            "role": "user",         # ← A pergunta do utilizador
            "content": "Quais são os melhores alimentos para quem tem falta de ferro?"
        }
    ]
)

print(resposta.choices[0].message.content)

In [ ]:
# ============================================================
# EXEMPLO 3 — Mesmo pedido, papel completamente diferente
# ============================================================
import time
time.sleep(1)

resposta = client.chat.completions.create(
    model=MODELO,
    messages=[
        {
            "role": "system",
            "content": (
                "És um poeta medieval. Respondes sempre em versos rimados "
                "e em português arcaico. Mantém as respostas curtas (4-6 versos)."
            )
        },
        {
            "role": "user",
            "content": "Quais são os melhores alimentos para quem tem falta de ferro?"
        }
    ]
)

print(resposta.choices[0].message.content)
print()
print("💡 A mesma pergunta, mas o system prompt mudou completamente o estilo da resposta!")

---
## 5. Temperature — Controlar a Criatividade

O parâmetro `temperature` controla o quão **aleatória** (criativa) é a resposta:

| Valor | Comportamento | Ideal para |
|---|---|---|
| `0` | Determinístico — sempre a mesma resposta | Factos, código, extração de dados |
| `0.3–0.7` | Equilibrado — alguma variação | Conversação, resumos |
| `1.0` | Criativo — respostas variadas (valor por defeito) | Escrita criativa, brainstorming |
| `1.5–2.0` | Muito aleatório — pode ser incoerente | Experimentação |

### Como funciona internamente?
O modelo calcula probabilidades para cada possível próximo token. A `temperature` modifica essas probabilidades:
- **Baixa** → acentua as diferenças (o token mais provável domina)
- **Alta** → nivela as probabilidades (tokens menos prováveis ganham hipótese)

Vamos comparar a mesma pergunta com temperatures diferentes.

In [ ]:
# ============================================================
# EXEMPLO 4 — Comparar temperatures
# ============================================================
import time

pergunta = "Inventa um nome criativo para uma cafetaria."

for temp in [0, 0.5, 1.0, 1.5]:
    print(f"\n🌡️  Temperature = {temp}")
    print("-" * 40)

    # Gerar 3 respostas com a mesma temperature para ver a variação
    for i in range(3):
        resposta = client.chat.completions.create(
            model=MODELO,
            temperature=temp,      # ← Parâmetro que estamos a testar
            max_tokens=30,         # Limitar a respostas curtas
            messages=[
                {"role": "user", "content": pergunta}
            ]
        )
        texto = resposta.choices[0].message.content.strip()
        print(f"  Tentativa {i+1}: {texto}")
        time.sleep(1)  # Pausa para respeitar rate limits

print("\n💡 Observa: com temperature=0 as respostas são iguais; com 1.5 são muito diferentes.")

---
## 6. Max Tokens — Limitar o Comprimento

O `max_tokens` define o **número máximo de tokens** que o modelo pode gerar na resposta.

### O que é um token?
- Um token não é exatamente uma palavra. É um pedaço de texto que o modelo processa.
- Em inglês: 1 token ≈ 0.75 palavras
- Em português: 1 token ≈ 0.5–0.6 palavras (acentos e caracteres especiais gastam mais tokens)
- Exemplo: "Olá, como estás?" → provavelmente 6-8 tokens

### Comportamento
- Se o modelo terminar antes do limite → `finish_reason = "stop"`
- Se atingir o limite → `finish_reason = "length"` (resposta cortada!)
- Mais tokens = resposta mais longa = **mais custo** em APIs pagas

In [ ]:
# ============================================================
# EXEMPLO 5 — Efeito do max_tokens
# ============================================================
import time

for max_t in [20, 50, 200]:
    resposta = client.chat.completions.create(
        model=MODELO,
        max_tokens=max_t,          # ← Parâmetro que estamos a testar
        messages=[
            {"role": "user", "content": "Explica o que é machine learning."}
        ]
    )
    texto = resposta.choices[0].message.content
    razao = resposta.choices[0].finish_reason
    tokens_usados = resposta.usage.completion_tokens

    print(f"\n📏 max_tokens = {max_t} | Tokens gerados: {tokens_usados} | Parou porque: {razao}")
    print("-" * 60)
    print(texto)
    time.sleep(1)

---
## 7. Conversa Multi-turno — Memória Artificial

Os LLMs **não têm memória entre pedidos**. Para simular uma conversa contínua, enviamos **todo o histórico** em cada pedido.

### Estrutura de uma conversa multi-turno:
```python
messages = [
    {"role": "system",    "content": "instruções globais"},
    {"role": "user",      "content": "primeira pergunta"},
    {"role": "assistant", "content": "primeira resposta do modelo"},
    {"role": "user",      "content": "segunda pergunta"},
    # → o modelo gera a segunda resposta com base em TODO este contexto
]
```

### Porque é necessário?
Quando o modelo recebe a segunda pergunta "E quais foram as causas?", precisa de saber que estávamos a falar da Primeira Guerra Mundial. Sem o histórico, não saberia interpretar o "E" nem "as causas" de quê.

### Implicação prática:
Conversas longas gastam **cada vez mais tokens** (e dinheiro), porque o histórico cresce a cada turno.

In [ ]:
# ============================================================
# EXEMPLO 6 — Conversa multi-turno
# ============================================================
import time

# Começamos com o system prompt
historico = [
    {
        "role": "system",
        "content": "És um professor de história. Respondes de forma clara e concisa em português."
    }
]

# Sequência de perguntas que dependem do contexto anterior
perguntas = [
    "Quando começou a Primeira Guerra Mundial?",
    "E quais foram as principais causas?",          # "E" → depende da pergunta anterior
    "Qual foi o papel de Portugal nesse conflito?"   # "nesse" → refere-se à guerra
]

for pergunta in perguntas:
    print(f"\n🧑 Utilizador: {pergunta}")

    # 1. Adicionar a pergunta ao histórico
    historico.append({"role": "user", "content": pergunta})

    # 2. Enviar TODO o histórico ao modelo
    resposta = client.chat.completions.create(
        model=MODELO,
        messages=historico,       # ← Histórico completo!
        max_tokens=200
    )

    texto = resposta.choices[0].message.content
    print(f"🤖 Modelo: {texto}")

    # 3. Guardar a resposta no histórico para o próximo turno
    historico.append({"role": "assistant", "content": texto})
    time.sleep(1)

print(f"\n📊 Mensagens no histórico: {len(historico)}")
print(f"   Total de caracteres: {sum(len(m['content']) for m in historico)}")
print("💡 Cada novo pedido envia todo este histórico — por isso conversas longas gastam mais tokens.")

---
## 8. Função Reutilizável

Para evitar repetir código, criamos uma função auxiliar que encapsula a chamada à API.

### Parâmetros da função `perguntar()`

| Parâmetro | Tipo | Default | Descrição |
|---|---|---|---|
| `pergunta` | `str` | — (obrigatório) | O texto da pergunta do utilizador. |
| `sistema` | `str` ou `None` | `None` | System prompt opcional para definir o papel do modelo. Se `None`, não é enviado system prompt. |
| `temperatura` | `float` | `1.0` | Grau de criatividade (0 = preciso, 2 = muito criativo). |
| `max_tokens` | `int` | `500` | Limite máximo de tokens na resposta. |

In [ ]:
# ============================================================
# Função reutilizável para chamar a API
# ============================================================

def perguntar(pergunta, sistema=None, temperatura=1.0, max_tokens=500):
    """
    Envia uma pergunta ao LLM e devolve a resposta como texto.

    Args:
        pergunta (str):      Texto da pergunta do utilizador.
        sistema (str):       System prompt para definir o papel do modelo. (opcional)
        temperatura (float): Criatividade da resposta, de 0 a 2. (default: 1.0)
        max_tokens (int):    Limite de tokens na resposta. (default: 500)

    Returns:
        str: Texto da resposta do modelo.
    """
    # Construir a lista de mensagens
    messages = []

    if sistema:  # Só adiciona system prompt se for fornecido
        messages.append({"role": "system", "content": sistema})

    messages.append({"role": "user", "content": pergunta})

    # Fazer o pedido à API
    resposta = client.chat.completions.create(
        model=MODELO,
        messages=messages,
        temperature=temperatura,
        max_tokens=max_tokens
    )

    return resposta.choices[0].message.content


print("✅ Função perguntar() definida!")

In [ ]:
# ============================================================
# EXEMPLO 7 — Mesma pergunta, papéis diferentes
# ============================================================
import time

pergunta = "Explica o que é a fotossíntese."

# Para uma criança de 8 anos
print("👧 Para uma criança:")
print(perguntar(
    pergunta,
    sistema="Explica como se falasses com uma criança de 8 anos. Usa exemplos do dia-a-dia. Máximo 3 frases.",
    max_tokens=150
))

time.sleep(1)

print("\n" + "=" * 60 + "\n")

# Para um estudante universitário
print("🎓 Para um universitário:")
print(perguntar(
    pergunta,
    sistema="Responde como professor universitário de biologia. Usa terminologia científica precisa. Máximo 3 frases.",
    max_tokens=150
))

---
## 9. Técnicas de Prompting na Prática

Vamos aplicar as três técnicas principais:

| Técnica | Descrição | Quando usar |
|---|---|---|
| **Zero-shot** | Pedir sem dar exemplos | Tarefas simples que o modelo já sabe fazer |
| **Few-shot** | Incluir 2-5 exemplos no prompt | Quando queremos um formato ou critério específico |
| **Chain-of-Thought** | Pedir raciocínio passo a passo | Problemas de lógica, matemática, decisões complexas |

In [ ]:
# ============================================================
# EXEMPLO 8a — Zero-shot (sem exemplos)
# ============================================================

print("🔵 ZERO-SHOT — o modelo usa apenas o seu conhecimento:")
print("-" * 50)
print(perguntar(
    "Classifica o sentimento da seguinte frase como Positivo, Negativo ou Neutro: "
    "'O restaurante era horrível, nunca mais volto.'",
    temperatura=0,
    max_tokens=50
))

In [ ]:
# ============================================================
# EXEMPLO 8b — Few-shot (com exemplos no prompt)
# ============================================================
import time
time.sleep(1)

prompt_fewshot = """Classifica o sentimento de cada frase como Positivo, Negativo ou Neutro.
Responde apenas com a classificação.

Frase: "Adorei a comida, estava deliciosa!"
Sentimento: Positivo

Frase: "O serviço foi péssimo e demorado."
Sentimento: Negativo

Frase: "O restaurante fica na Rua Augusta."
Sentimento: Neutro

Frase: "A decoração era bonita mas a comida estava fria."
Sentimento:"""

print("🟢 FEW-SHOT — o modelo segue o padrão dos exemplos:")
print("-" * 50)
print(perguntar(prompt_fewshot, temperatura=0, max_tokens=20))

In [ ]:
# ============================================================
# EXEMPLO 8c — Chain-of-Thought (raciocínio passo a passo)
# ============================================================
import time
time.sleep(1)

problema = """Uma loja tem 3 prateleiras. Cada prateleira tem 4 caixas.
Cada caixa tem 6 livros. A Maria comprou 5 livros.
Quantos livros ficaram na loja?

Pensa passo a passo antes de dar a resposta final."""

print("🟡 CHAIN-OF-THOUGHT — o modelo mostra o raciocínio:")
print("-" * 50)
print(perguntar(problema, temperatura=0, max_tokens=300))

---
## 10. Exercícios Práticos

Experimenta por ti! Modifica os exemplos acima ou tenta os seguintes desafios:

1. **Muda o system prompt** — faz o modelo responder como pirata, como advogado, ou como um robot
2. **Compara temperatures** — pede uma receita de bolo com `temperature=0` vs `temperature=1.5`
3. **Testa o few-shot** — cria exemplos para o modelo classificar emails como "urgente" ou "pode esperar"
4. **Testa os limites** — o que acontece se puseres `max_tokens=5`?
5. **Conversa multi-turno** — adapta o Exemplo 6 para um tema diferente (ex: ciência, culinária)

In [ ]:
# ============================================================
# ESPAÇO PARA EXPERIMENTAR
# ============================================================

# Escreve o teu código aqui!
# Usa a função perguntar() para simplificar:
#
# print(perguntar(
#     "A tua pergunta aqui",
#     sistema="O papel do modelo",
#     temperatura=0.5,
#     max_tokens=200
# ))


---
## Resumo

| Conceito | O que aprendemos |
|---|---|
| `client = OpenAI(api_key, base_url)` | Criar o cliente que se liga à API. A `base_url` define o fornecedor. |
| `messages` | A conversa é uma lista de dicionários, cada um com `role` e `content`. |
| `role: system` | Define o papel e comportamento do modelo (invisível ao utilizador). |
| `role: user` | A mensagem/pergunta do utilizador. |
| `role: assistant` | Resposta anterior do modelo (para multi-turno). |
| `temperature` | Controla a criatividade (0 = preciso, 1+ = criativo). |
| `max_tokens` | Limita o comprimento da resposta. |
| Multi-turno | Enviar todo o histórico em cada pedido para simular memória. |
| Zero-shot | Pedir sem dar exemplos. |
| Few-shot | Incluir exemplos no prompt para guiar o formato. |
| Chain-of-Thought | Pedir "pensa passo a passo" para problemas de raciocínio. |

---
## Fornecedores Alternativos (todos gratuitos)

Para mudar de fornecedor, basta alterar `API_KEY`, `BASE_URL` e `MODELO` na Secção 2:

| Fornecedor | `base_url` | Modelo recomendado | Limites gratuitos |
|---|---|---|---|
| **Groq** | `https://api.groq.com/openai/v1` | `llama-3.3-70b-versatile` | ~1000 req/dia, 500k tokens/min |
| **Google Gemini** | `https://generativelanguage.googleapis.com/v1beta/openai/` | `gemini-2.0-flash` | ~20 req/dia (limites reduzidos) |
| **OpenRouter** | `https://openrouter.ai/api/v1` | `google/gemini-2.0-flash-exp:free` | ~50 req/dia |

### Obter chaves:
- **Groq:** [console.groq.com/keys](https://console.groq.com/keys)
- **Google:** [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)
- **OpenRouter:** [openrouter.ai/keys](https://openrouter.ai/keys)